# SmolDocling Tutorial for Document Processing & RAG Chunking

Vision-Language Models (VLM) recently became a big thing. Let's take a look how we can use them in document parsing.

This notebook demonstrates:

1. Installing and loading SmolDocling  
2. Converting a PDF into a DoclingDocument  
3. Extracting structure (e.g. table of contents)  
4. Chunking the document into smaller pieces suitable for Retrieval-Augmented Generation (RAG)  
5. Saving the chunks to `data/preprocessed_SmolDocling/`

## Imports and Device Configuration

In [60]:
from pprint import pprint
import os
import torch
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from docling.document_converter import DocumentConverter

from transformers import AutoTokenizer
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")


Using device: cpu


## Load and Convert Your Document

In [61]:
# Path to your PDF (can also be a URL)
source = "/Users/philipp.schwartenbeck/Desktop/rag-llm-applications/data/raw/Geschäftsordnung  Trust Council.pdf"

# Initialize the converter and run
converter = DocumentConverter()
result = converter.convert(source)

# The converted document
doc: DoclingDocument = result.document

Task 1.1 Inspect doc and try it with different documents and document types! What do you see?

## Inspecting the Document

### Export as Markdown

In [62]:
md = doc.export_to_markdown()
print(md[:1200])  # print the first 500 characters

## Geschäftsordnung Trust Council [at]

Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen:

## § 1 Zweck

Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.

## § 2 Aufgaben des TC-Vorsitzenden

Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und übernimmt die Umsetzung von Beschlüssen.

Dazu gehören beispielsweise:

- · die Vorbereitung von Sitzungen
- · die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen
- · der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren Parteien
- · die Vertretung des Trust Council nach außen

Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden.

## § 3 Vertretung des/der TC-Vorsitzenden

- 1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvertretende TCVorsitzend

### Export as Dict (Structured)

In [63]:
doc_dict = doc.export_to_dict()

pprint(doc_dict)

{'body': {'children': [{'$ref': '#/texts/0'},
                       {'$ref': '#/texts/1'},
                       {'$ref': '#/texts/2'},
                       {'$ref': '#/texts/3'},
                       {'$ref': '#/texts/4'},
                       {'$ref': '#/texts/5'},
                       {'$ref': '#/texts/6'},
                       {'$ref': '#/groups/0'},
                       {'$ref': '#/texts/11'},
                       {'$ref': '#/texts/12'},
                       {'$ref': '#/groups/1'},
                       {'$ref': '#/texts/15'},
                       {'$ref': '#/texts/16'},
                       {'$ref': '#/groups/2'},
                       {'$ref': '#/texts/18'},
                       {'$ref': '#/texts/19'},
                       {'$ref': '#/groups/3'},
                       {'$ref': '#/texts/22'},
                       {'$ref': '#/groups/4'},
                       {'$ref': '#/texts/29'},
                       {'$ref': '#/groups/5'},
                    

In [64]:
print("doc_dict keys:", doc_dict.keys())

doc_dict keys: dict_keys(['schema_name', 'version', 'name', 'origin', 'furniture', 'body', 'groups', 'texts', 'pictures', 'tables', 'key_value_items', 'form_items', 'pages'])


Task 1.2 Inspect the Content of this dict!

In [65]:
# show the first 10 text objects
import json
for i, txt in enumerate(doc_dict["texts"][:10]):
    print(f"--- text #{i} ---")
    print(json.dumps(txt, indent=2))


--- text #0 ---
{
  "self_ref": "#/texts/0",
  "parent": {
    "$ref": "#/body"
  },
  "children": [],
  "content_layer": "body",
  "label": "section_header",
  "prov": [
    {
      "page_no": 1,
      "bbox": {
        "l": 70.824,
        "t": 765.43,
        "r": 455.403,
        "b": 737.47,
        "coord_origin": "BOTTOMLEFT"
      },
      "charspan": [
        0,
        35
      ]
    }
  ],
  "orig": "Gesch\u00e4ftsordnung Trust Council [at]",
  "text": "Gesch\u00e4ftsordnung Trust Council [at]",
  "level": 1
}
--- text #1 ---
{
  "self_ref": "#/texts/1",
  "parent": {
    "$ref": "#/body"
  },
  "children": [],
  "content_layer": "body",
  "label": "text",
  "prov": [
    {
      "page_no": 1,
      "bbox": {
        "l": 70.824,
        "t": 706.18,
        "r": 526.989,
        "b": 679.66,
        "coord_origin": "BOTTOMLEFT"
      },
      "charspan": [
        0,
        174
      ]
    }
  ],
  "orig": "Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in sein

## Extracting the Table of Contents

In [66]:
# Get individual text blocks
raw_texts = doc_dict["texts"]

raw_texts

[{'self_ref': '#/texts/0',
  'parent': {'$ref': '#/body'},
  'children': [],
  'content_layer': 'body',
  'label': 'section_header',
  'prov': [{'page_no': 1,
    'bbox': {'l': 70.824,
     't': 765.43,
     'r': 455.403,
     'b': 737.47,
     'coord_origin': 'BOTTOMLEFT'},
    'charspan': [0, 35]}],
  'orig': 'Geschäftsordnung Trust Council [at]',
  'text': 'Geschäftsordnung Trust Council [at]',
  'level': 1},
 {'self_ref': '#/texts/1',
  'parent': {'$ref': '#/body'},
  'children': [],
  'content_layer': 'body',
  'label': 'text',
  'prov': [{'page_no': 1,
    'bbox': {'l': 70.824,
     't': 706.18,
     'r': 526.989,
     'b': 679.66,
     'coord_origin': 'BOTTOMLEFT'},
    'charspan': [0, 174]}],
  'orig': 'Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen:',
  'text': 'Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 m

Check what the texts contain:

In [67]:
for t in raw_texts:
    print(t["text"])
    print("===")

Geschäftsordnung Trust Council [at]
===
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen:
===
§ 1 Zweck
===
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen.
===
§ 2 Aufgaben des TC-Vorsitzenden
===
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und übernimmt die Umsetzung von Beschlüssen.
===
Dazu gehören beispielsweise:
===
· die Vorbereitung von Sitzungen
===
· die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen
===
· der anfallende Schriftwechsel mit Geschäftsleitung, juristischen Beratern sowie weiteren Parteien
===
· die Vertretung des Trust Council nach außen
===
Die obigen Aufgabengebiete können auch an andere Mitglieder des TC delegiert werden.
===
§ 3 Vertretung des/der TC-Vorsitzenden
===
1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvert

Check labels of raw texts

In [68]:
for t in raw_texts:
    print(t.get("label"))

section_header
text
section_header
text
section_header
text
text
list_item
list_item
list_item
list_item
text
section_header
list_item
list_item
section_header
text
list_item
text
section_header
list_item
list_item
page_footer
list_item
list_item
list_item
list_item
list_item
list_item
section_header
list_item
list_item
list_item
list_item
list_item
section_header
list_item
list_item
section_header
list_item
list_item
list_item
text
list_item
list_item
list_item
list_item
list_item
section_header
list_item
list_item
list_item
section_header
list_item
list_item
list_item
list_item
list_item
section_header
text
section_header
text
section_header
text
text


Now, print the table of contents

In [69]:
header_idxs = [i for i, t in enumerate(raw_texts) if t.get("label") == "section_header"]

for chapter, idx in enumerate(header_idxs):
    print(f"{chapter+1}. {raw_texts[idx]['text']}")

1. Geschäftsordnung Trust Council [at]
2. § 1 Zweck
3. § 2 Aufgaben des TC-Vorsitzenden
4. § 3 Vertretung des/der TC-Vorsitzenden
5. § 4 Mitglieder des Trust Council
6. § 5 Arbeitsplanung des TC
7. § 7 Einladung zur TC-Sitzung
8. § 8 Verlauf der TC-Sitzung
9. § 9 Beschlussfassung des TC
10. § 10 Niederschrift
11. § 11 Mitarbeiterversammlung
12. § 12 Bekanntmachungen
13. § 13 Wahl zur nächsten Amtsperiode
14. § 14 Laufzeit


## Chunking the Document for RAG

Build sections based on headers - What is going on here?

In [70]:
sections = []
for idx, start in enumerate(header_idxs):
    # determine end of this section
    end = header_idxs[idx + 1] if idx + 1 < len(header_idxs) else len(raw_texts)
    
    # title is the header text 
    title = raw_texts[start]["text"].strip()
    
    # collect all body blocks (text + list_item) under this header
    body_parts = []
    for t in raw_texts[start + 1 : end]:
        lbl = t.get("label")
        if lbl in ("text", "list_item"):
            # optionally, for list_item you can add a bullet:
            if lbl == "list_item":
                body_parts.append(f"- {t['text'].strip()}")
            else:
                body_parts.append(t["text"].strip())
    
    full_text = "\n\n".join(body_parts)
    sections.append({"title": title, "text": full_text})

print(f"Built {len(sections)} sections via headers.")

Built 14 sections via headers.


Adjust max_tokens to taste

In [71]:
tokenizer = AutoTokenizer.from_pretrained("ds4sd/SmolDocling-256M-preview")

In [72]:
out_dir = "data/preprocessed_SmolDocling"
os.makedirs(out_dir, exist_ok=True)

In [73]:

chunked_files = []

for sec_idx, sec in enumerate(sections):
    tokens = tokenizer.tokenize(sec["text"])
    print(f"Section {sec_idx+1} has {len(tokens)} tokens.")
    chunk_text   = tokenizer.convert_tokens_to_string(tokens)
    
    fname        = f"section{sec_idx+1}.txt"
    path         = os.path.join(out_dir, fname)
    
    with open(path, "w", encoding="utf-8") as f:
        f.write(chunk_text)
    chunked_files.append(path)

print(f"Saved {len(chunked_files)} chunks under `{out_dir}`")

Section 1 has 65 tokens.
Section 2 has 29 tokens.
Section 3 has 196 tokens.
Section 4 has 127 tokens.
Section 5 has 207 tokens.
Section 6 has 555 tokens.
Section 7 has 348 tokens.
Section 8 has 226 tokens.
Section 9 has 616 tokens.
Section 10 has 251 tokens.
Section 11 has 249 tokens.
Section 12 has 67 tokens.
Section 13 has 68 tokens.
Section 14 has 152 tokens.
Saved 14 chunks under `data/preprocessed_SmolDocling`


Task 1.3 In the above loop, implement a way of controlling a specific chunk size!

In [74]:
max_tokens = 500
chunked_files = []

for sec_idx, sec in enumerate(sections):
    tokens = tokenizer.tokenize(sec["text"])
    for i in range(0, len(tokens), max_tokens):
        chunk_tokens = tokens[i : i + max_tokens]
        chunk_text   = tokenizer.convert_tokens_to_string(chunk_tokens)
        fname        = f"section{sec_idx+1}_chunk{(i//max_tokens)+1}.txt"
        path         = os.path.join(out_dir, fname)
        with open(path, "w", encoding="utf-8") as f:
            f.write(chunk_text)
        chunked_files.append(path)

print(f"Saved {len(chunked_files)} chunks under `{out_dir}`")

Saved 16 chunks under `data/preprocessed_SmolDocling`


Now save as a json and txt:

In [75]:
chunk_out = "data/preprocessed_SmolDocling/chunks"
os.makedirs(chunk_out, exist_ok=True)

for path in chunked_files:
    # read the text again
    with open(path, encoding="utf-8") as f:
        chunk_text = f.read()
    # prepare metadata
    meta = {
        "chunk_filename": os.path.basename(path),
        "text": chunk_text
    }
    base = os.path.splitext(os.path.basename(path))[0]
    # 1) JSON
    with open(os.path.join(chunk_out, f"{base}.json"), "w", encoding="utf-8") as jf:
        json.dump(meta, jf, ensure_ascii=False, indent=2)
    # 2) TXT (just the raw chunk)
    with open(os.path.join(chunk_out, f"{base}.txt"), "w", encoding="utf-8") as tf:
        tf.write(chunk_text)

print(f"Saved {len(chunked_files)} chunks as JSON+TXT in `{chunk_out}`")

Saved 16 chunks as JSON+TXT in `data/preprocessed_SmolDocling/chunks`


### Verifying Your Chunks

In [76]:
# List first few chunk files and display their content
for p in chunked_files[:5]:
    print(f"\n--- {p} ---")
    print(open(p, encoding="utf-8").read()[:300], "...\n")


--- data/preprocessed_SmolDocling/section1_chunk1.txt ---
Der Trust Council (TC) der Firma Alexander Thamm GmbH hat in seiner Sitzung am 16.11.2023 mit der Stimmenmehrheit seiner Mitglieder nachfolgende Geschäftsordnung beschlossen: ...


--- data/preprocessed_SmolDocling/section2_chunk1.txt ---
Die Geschäftsordnung wurde beschlossen um den Arbeitsmodus des TC festzulegen. ...


--- data/preprocessed_SmolDocling/section3_chunk1.txt ---
Soweit nicht anders vereinbart, führt der/die Vorsitzende des TC die laufenden Geschäfte und übernimmt die Umsetzung von Beschlüssen.

Dazu gehören beispielsweise:

- · die Vorbereitung von Sitzungen

- · die Erteilung und der Entzug des Rederechts in Trust Council Sitzungen

- · der anfallende Schr ...


--- data/preprocessed_SmolDocling/section4_chunk1.txt ---
- 1. Im Falle der Verhinderung des/der TC-Vorsitzenden nimmt der/die stellvertretende TCVorsitzende seine/ihre Aufgaben wahr.

- 2. Ist die Übernahme der Aufgaben aufgrund einer Verhinderung auc